# Two-Stage Oral Cavity Pipeline (Binary → 4-Class + Unknown)

This notebook builds a **two-model pipeline**:
1) **Binary** cancer vs non-cancer (binary dataset)
2) **4-class** (Benign, Healthy, OCA, OPMD) from the 5-class dataset **excluding Unknown**

Unknown handling is done at inference: if the 4-class max prob is below a threshold, emit `Unknown`.

Evaluation compares against the same test set used by the ensemble baseline (CONFIG['ensemble_test_dir']).

Set `train_two_stage=True` to train the two small models; otherwise place pre-trained models at:
- `results/multihead_efficientnet/<run>/models/binary_stage.keras`
- `results/multihead_efficientnet/<run>/models/four_class_stage.keras`

Runs on Apple Silicon-friendly optimizers (legacy AdamW).

In [11]:
# Cell 1: Config and imports
import os, json, random, math
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tqdm import tqdm
import cv2
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

CONFIG = {
    'data_dir_5class': os.path.expanduser('~/dev/oral_cavity_5class_dataset'),
    'data_dir_binary': os.path.expanduser('~/dev/oral_cavity_binary_dataset'),
    'ensemble_test_dir': os.path.expanduser('~/dev/oral_cavity_5class_dataset/test'),
    'image_size': (224, 224),
    'batch_size': 32,
    'learning_rate': 5e-5,
    'weight_decay': 1e-4,
    'class_names': ['Benign', 'Healthy', 'OCA', 'OPMD', 'Unknown'],
}

output_root = Path('results/two_stage_pipeline')
output_root.mkdir(parents=True, exist_ok=True)
output_models = output_root / 'models'
output_models.mkdir(exist_ok=True)

binary_model_path = output_models / 'binary_stage.keras'
four_class_model_path = output_models / 'four_class_stage.keras'

AUTOTUNE = tf.data.AUTOTUNE

def load_image_paths_and_labels(data_dir: str, split: str, class_names):
    base_path = Path(data_dir) / split
    image_paths, labels = [], []
    for idx, name in enumerate(class_names):
        class_path = base_path / name
        if class_path.exists():
            for ext in ['*.jpg','*.jpeg','*.png']:
                for p in class_path.glob(ext):
                    image_paths.append(str(p))
                    labels.append(idx)
    return image_paths, labels

def load_binary_dataset(data_dir: str):
    base = Path(data_dir)
    paths, labels = [], []
    for name, lab in [('non-cancer',0), ('cancer',1)]:
        d = base / name
        if d.exists():
            for ext in ['*.jpg','*.jpeg','*.png']:
                for p in d.glob(ext):
                    paths.append(str(p)); labels.append(lab)
    return paths, labels

def load_and_preprocess_image(path, image_size):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, image_size)
    return img

print('Config loaded; output to', output_root)

Config loaded; output to results/two_stage_pipeline


In [12]:
# Cell 2: Build datasets (4-class excl. Unknown, and binary)

# 5-class data
train_paths_5, train_labels_5 = load_image_paths_and_labels(CONFIG['data_dir_5class'], 'train', CONFIG['class_names'])
val_paths_5, val_labels_5 = load_image_paths_and_labels(CONFIG['data_dir_5class'], 'val', CONFIG['class_names'])
test_paths_5, test_labels_5 = load_image_paths_and_labels(CONFIG['data_dir_5class'], 'test', CONFIG['class_names'])

def filter_unknown(paths, labels):
    p_out, l_out = [], []
    for p,l in zip(paths, labels):
        if l == 4: # Unknown
            continue
        # remap 0..3
        l_out.append(l)
        p_out.append(p)
    return p_out, l_out

train_paths_4, train_labels_4 = filter_unknown(train_paths_5, train_labels_5)
val_paths_4, val_labels_4 = filter_unknown(val_paths_5, val_labels_5)
test_paths_4, test_labels_4 = filter_unknown(test_paths_5, test_labels_5)

# Binary data
bin_paths, bin_labels = load_binary_dataset(CONFIG['data_dir_binary'])

def make_ds(paths, labels, batch_size, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths))
    def py_map(path, label):
        img = load_and_preprocess_image(path.numpy().decode(), CONFIG['image_size']).astype(np.float32)/255.0
        return img, tf.cast(label, tf.int32)
    def tf_map(path, label):
        img, lab = tf.py_function(py_map, [path, label], [tf.float32, tf.int32])
        img.set_shape((*CONFIG['image_size'],3)); lab.set_shape([])
        return img, lab
    ds = ds.map(tf_map, num_parallel_calls=AUTOTUNE).batch(batch_size).prefetch(AUTOTUNE)
    return ds

batch = CONFIG['batch_size']
train_ds_bin = make_ds(bin_paths, bin_labels, batch, shuffle=True)
val_ds_bin = make_ds(val_paths_5, [1 if l in [2,3] else 0 for l in val_labels_5], batch, shuffle=False)
train_ds_4 = make_ds(train_paths_4, train_labels_4, batch, shuffle=True)
val_ds_4 = make_ds(val_paths_4, val_labels_4, batch, shuffle=False)

print('Binary samples:', len(bin_paths))
print('4-class samples (train):', len(train_paths_4))

Binary samples: 1634
4-class samples (train): 2357


In [13]:
# Cell 3: Build/Train two small models (always train)

def build_binary_model(input_shape):
    inp = layers.Input(shape=input_shape)
    x = EfficientNetB0(include_top=False, weights='imagenet', pooling='avg')(inp)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation='sigmoid', dtype='float32')(x)
    return models.Model(inp, out, name='binary_cancer_model')

def build_four_class_model(input_shape):
    inp = layers.Input(shape=input_shape)
    x = EfficientNetB0(include_top=False, weights='imagenet', pooling='avg')(inp)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(4, activation='softmax', dtype='float32')(x)
    return models.Model(inp, out, name='four_class_model')

# Binary
bin_model = build_binary_model((*CONFIG['image_size'],3))
bin_model.compile(
    optimizer=tf.keras.optimizers.legacy.Adam(learning_rate=CONFIG['learning_rate']),
    loss='binary_crossentropy', metrics=['accuracy']
)
bin_ckpt = ModelCheckpoint(binary_model_path, save_best_only=True, monitor='val_loss', mode='min')
bin_es = EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss', mode='min')
bin_model.fit(train_ds_bin, validation_data=val_ds_bin, epochs=30, callbacks=[bin_ckpt, bin_es], verbose=2)

# Four-class
four_model = build_four_class_model((*CONFIG['image_size'],3))
four_model.compile(
    optimizer=tf.keras.optimizers.legacy.Adam(learning_rate=CONFIG['learning_rate']),
    loss='sparse_categorical_crossentropy', metrics=['accuracy']
)
four_ckpt = ModelCheckpoint(four_class_model_path, save_best_only=True, monitor='val_loss', mode='min')
four_es = EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss', mode='min')
four_model.fit(train_ds_4, validation_data=val_ds_4, epochs=30, callbacks=[four_ckpt, four_es], verbose=2)

print('Two-stage training complete. Models saved to:', binary_model_path, 'and', four_class_model_path)

AttributeError: 'PosixPath' object has no attribute 'endswith'

In [ ]:
# Cell 4: Two-stage inference + eval on ensemble test set
cancer_gate = 0.5
unknown_thresh = 0.55

def two_stage_predict(batch_images, binary_model, four_model, cancer_gate=0.5, unknown_thresh=0.55):
    bin_prob = binary_model(batch_images, training=False).numpy().squeeze()
    four_probs = four_model(batch_images, training=False).numpy()
    final_preds = []
    for i in range(len(batch_images)):
        p_bin = bin_prob[i]
        p_four = four_probs[i]
        top_idx = int(np.argmax(p_four)); top_conf = float(np.max(p_four))
        if top_conf < unknown_thresh:
            final_preds.append(4); continue
        # cancer gate
        if p_bin >= cancer_gate:
            if top_idx in [2,3]:
                final_preds.append(top_idx)
            else:
                final_preds.append(3)  # high-risk fallback
        else:
            if top_idx in [0,1]:
                final_preds.append(top_idx)
            else:
                final_preds.append(4)  # Unknown on conflict
    return np.array(final_preds)

print('\nRunning two-stage evaluation...')
if not binary_model_path.exists() or not four_class_model_path.exists():
    print(f"❌ Models missing. Train them (set train_two_stage=True) or place models at:\n  {binary_model_path}\n  {four_class_model_path}")
else:
    bin_model = keras.models.load_model(binary_model_path, compile=False)
    four_model = keras.models.load_model(four_class_model_path, compile=False)

    # Use ensemble test set (5-class)
    test_paths = test_paths_5
    test_labels = test_labels_5
    bsz = CONFIG['batch_size']
    num_batches = int(np.ceil(len(test_paths)/bsz))
    all_preds, all_labels = [], []
    for b in tqdm(range(num_batches), desc='Two-stage eval'):
        s,e = b*bsz, min((b+1)*bsz, len(test_paths))
        batch_imgs = [load_and_preprocess_image(p, CONFIG['image_size']).astype(np.float32)/255.0 for p in test_paths[s:e]]
        batch_imgs = np.stack(batch_imgs, axis=0)
        preds = two_stage_predict(batch_imgs, bin_model, four_model, cancer_gate, unknown_thresh)
        all_preds.extend(preds.tolist()); all_labels.extend(test_labels[s:e])

    all_preds = np.array(all_preds); all_labels = np.array(all_labels)
    overall_acc = float(np.mean(all_preds == all_labels))
    print('\n📊 Two-Stage Results:')
    print(f'   Total samples: {len(all_labels)}')
    print(f'   Overall accuracy: {overall_acc:.4f}')

    print('\n📈 Per-class accuracy:')
    for i,name in enumerate(CONFIG['class_names']):
        m = all_labels==i
        acc = float(np.mean(all_preds[m]==all_labels[m])) if m.sum()>0 else float('nan')
        print(f'   {name}: {acc:.4f}')

    # Prediction distribution diagnostic
    up, counts = np.unique(all_preds, return_counts=True)
    print('\n🔍 Prediction distribution:')
    for u,c in zip(up, counts):
        pct = c/len(all_preds)*100
        print(f'   {CONFIG['class_names'][u]}: {c} ({pct:.1f}%)')
    if max(counts)/len(all_preds) > 0.9:
        print('⚠️  Model appears collapsed (predicting one class >90%).')
